# Imports

In [1]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, roc_curve, auc

from autogluon.tabular import TabularDataset, TabularPredictor
#from autogluon.multimodal import MultiModalPredictor
import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

VAL_TILE_FRACTION = 0.4
TILE_SIZE_DEG = 0.01  # approx 1km


SCORING = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='r2',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__fit_intercept': [True, False],
    'model__positive': [False, True]
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}

def evaluate_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    errors = y_true - y_pred

    bin_size = 0.1
    bins = np.arange(0.0, 1.0 + bin_size, bin_size)
    bin_ids = np.digitize(y_pred, bins, right=False)

    ece = 0.0
    N = len(y_pred)

    for i in range(1, len(bins)):
        mask = bin_ids == i
        if not np.any(mask):
            continue

        bin_center = (bins[i - 1] + bins[i]) / 2.0
        median_obs = np.median(y_true[mask])
        ece += (mask.sum() / N) * abs(median_obs - bin_center)
        
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': float(np.mean(np.abs(errors))),
        'mse': float(np.mean(errors ** 2)),
        "p95": np.quantile(np.abs(errors), 0.95),
        'ece': float(ece),              # Expected calibration error
    }


def record_regression_results(results, target, model_name, y_true, y_pred, split='test'):
    metrics = evaluate_regression_metrics(y_true, y_pred)
    split_label = split.capitalize()
    print(f"{split_label} R2 score {metrics['r2']:.4f}")
    print(f"{split_label} MAE score {metrics['mae']:.4f}")
    print(f"{split_label} MSE score {metrics['mse']:.4f}")
    print(f"{split_label} P95 score {metrics['p95']:.4f}")
    print(f"{split_label} ECE score {metrics['ece']:.4f}")
    results.append({
        'target': target,
        'model': model_name,
        f'{split}_r2': metrics['r2'],
        f'{split}_mae': metrics['mae'],
        #f'{split}_mse': metrics['mse'],
        #f'{split}_p95': metrics['p95'],
        f'{split}_ece': metrics['ece'],
    })
    return metrics


# Load data

In [2]:
model_det = "yolo"
data_path_file = f"{model_det}_metafeatures.csv"
data = pd.read_csv("../data/" + data_path_file, index_col=0)

print(data)

      country time_of_day        lat       long       road_type  \
1          PL         day  52.249511  21.043137            city   
6          PL         day  52.239332  21.030383            city   
39         PL         day  52.234241  21.003985            city   
49         PL         day  52.224666  21.071192            city   
52         PL    twilight  52.231355  21.054809            city   
...       ...         ...        ...        ...             ...   
99904      PL         day  54.353453  18.645450  arterial-urban   
99939      DE       night  50.116028   8.622851         highway   
99941      DE         day  53.057472   8.958626  arterial-urban   
99984      IT       night  41.935093  12.599486   smaller-rural   
99997      SE    twilight  59.103593  12.789612  arterial-rural   

      road_condition              weather  solar_angle_elevation  month  hour  \
1             normal    partly-cloudy-day              51.723833      4    10   
6             normal            c

In [3]:
model_all = pd.read_csv(f"../results/assessors/{model_det}_ass_results_table.csv")
base_iou_r2 = model_all.loc[(model_all['target'] == "IoU") & (model_all['model'] == "Autogluon_Tab"), 'test_r2'].values[0]
base_iou_ece = model_all.loc[(model_all['target'] == "IoU") & (model_all['model'] == "Autogluon_Tab"), 'test_ece'].values[0]
base_lrp_r2 = model_all.loc[(model_all['target'] == "LRP") & (model_all['model'] == "Autogluon_Tab"), 'test_r2'].values[0]
base_lrp_ece = model_all.loc[(model_all['target'] == "LRP") & (model_all['model'] == "Autogluon_Tab"), 'test_ece'].values[0]
print(f"Base R2: {base_iou_r2}, Base ECE: {base_iou_ece}")
print(f"Base R2: {base_lrp_r2}, Base ECE: {base_lrp_ece}")

Base R2: 0.4957, Base ECE: 0.0120996896393109
Base R2: 0.564, Base ECE: 0.0124041241264758


In [4]:
data = data.dropna()
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 9,655 rows and 42 columns


In [5]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["mean_conf"]
#image_paths_dict = {int(img_pth.split("_")[0]):f"../data/zod_yolo/images/test/{img_pth}" for img_pth in os.listdir("../data/zod_yolo/images/test") if img_pth.endswith(".jpg")}
#img_path = pd.DataFrame.from_dict(image_paths_dict, orient='index', columns=["Images"])

drop_columns = ["iou", "lrp"]
data = data.drop(columns=drop_columns, errors='ignore')
data_indices = data.index.to_numpy()

In [6]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 9655
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'cloud_cover', 'complexity', 'contrast',
       'country', 'distortion_magnitude', 'field_view_horizontal',
       'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat',
       'lateral_acceleration', 'lateral_velocity', 'long', 'max_conf',
       'mean_conf', 'min_conf', 'month', 'noisiness', 'num_detections',
       'quality', 'rain', 'relative_humidity_2m', 'road_condition',
       'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation',
       'std_conf', 'temperature_2m', 'time_of_day', 'weather',
       'wind_speed_10m'],
      dtype='object')
Number of columns: 37


In [7]:
numeric_columns = ['solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']
numeric_columns = numeric_columns + ['mean_conf', "std_conf", "max_conf", "min_conf", "num_detections"]


categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'max_conf', 'mean_conf', 'min_conf', 'month', 'noisiness', 'num_detections', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'std_conf', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


In [8]:
feature_groups = {
    "weather": ['solar_angle_elevation',  'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m' , 'weather'] ,
    "kinematic": ["forward_acceleration", "lateral_acceleration", "forward_velocity", "lateral_velocity"],
    "calibration": ['field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset'],
    "image_quality": ["laplacian", "brightness", "contrast", "sharpness", "noisiness", "quality", "complexity"],
    "model": ['mean_conf', "std_conf", "max_conf", "min_conf", "num_detections"],
    "location": ["lat", "long", "country"],
    "temporal": ["time_of_day", "road_type", "road_condition", "hour", "month"]
} 




# Assessor Tests

Split data into train 60% and validation 40%. To prevent leakage, the data is binned in buckets by lat/long in 1km. The split is done by buckets ensuring no leakage. 

In [9]:
#Val split
def tile_id(lat, lon, size_deg):
    lat_bin = np.floor(lat / size_deg) * size_deg
    lon_bin = np.floor(lon / size_deg) * size_deg
    return f'{lat_bin:.4f}_{lon_bin:.4f}'

In [10]:
## Make split of train into train and val
ids = data.index.tolist()

coords = {frame_id: {"lat": data.loc[frame_id, "lat"], "lon": data.loc[frame_id, "long"]} for frame_id in ids}
df_coords = pd.DataFrame.from_dict(coords, orient='index')
df_coords['tile_id'] = [tile_id(lat, lon, TILE_SIZE_DEG) for lat, lon in zip(df_coords['lat'], df_coords['lon'])]

unique_tiles = sorted(set(df_coords['tile_id'].dropna().unique()))
rng = np.random.default_rng(SEED)
val_tile_count = int(len(unique_tiles) * VAL_TILE_FRACTION)
val_tiles = set(rng.choice(unique_tiles, size=val_tile_count, replace=False))
df_coords['split'] = np.where(df_coords['tile_id'].isin(val_tiles), 'val', 'train')
test_idx = df_coords[df_coords['split'] == 'val'].index.tolist()
train_idx = df_coords[df_coords['split'] == 'train'].index.tolist()
print(f"Tile split -> train: {len(train_idx)}, val: {len(test_idx)}")

Tile split -> train: 5518, val: 4137


In [11]:
X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print("Conf:", len(conf_train), len(conf_test))
print(X_train.columns)

X: 5518 4137
y: 5518 4137
Conf: 5518 4137
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour',
       'forward_acceleration', 'lateral_acceleration', 'forward_velocity',
       'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m',
       'weather_code', 'num_detections', 'max_conf', 'min_conf', 'mean_conf',
       'std_conf'],
      dtype='object')


## Autogluon

In [12]:
model_results = []

In [13]:

def autoglue_regression(X_train, y_train, X_test, y_test, target_name):
    train = pd.merge(X_train, y_train, left_index=True, right_index=True)
    autoglue_model = TabularPredictor(label=target_name, problem_type="regression", eval_metric="r2", path=f"../models/assessors/autoglue_tab/{target_name}/", verbosity=0)
    autoglue_predictor = autoglue_model.fit(train)
    autoglue_predictor.fit_summary(verbosity=0)
    y_pred_autg = autoglue_predictor.predict(X_test)
    return y_pred_autg

### IoU

In [14]:
delta_results = []
for feature in feature_groups.keys():
    for target in ["iou", "lrp"]:
        if target == "iou":
            y_train_target = y_train
            y_test_target = y_test
        else:
            y_train_target = y_train_lrp
            y_test_target = y_test_lrp
        print(f"Evaluating feature when excluding: {feature}")
        features_to_not_use = feature_groups[feature]
        X_train_subset = X_train.drop(columns=features_to_not_use, errors='ignore')
        X_test_subset = X_test.drop(columns=features_to_not_use, errors='ignore')
        
        y_pred_autg = autoglue_regression(X_train_subset, y_train_target, X_test_subset, y_test_target, target_name=target)
        metrics = record_regression_results([], target=target, model_name=f"AutoGluon_{feature}", y_true=y_test_target, y_pred=y_pred_autg)
        delta_results.append({
            "target": target,
            "feature_group": feature,
            "delta_r2": metrics['r2'] - base_iou_r2,
            "delta_ece": metrics['ece'] - base_iou_ece,
        })

Evaluating feature when excluding: weather


		f-string expression part cannot include a backslash (docments.py, line 359)
/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Test R2 score 0.4948
Test MAE score 0.1064
Test MSE score 0.0193
Test P95 score 0.2902
Test ECE score 0.0109
Evaluating feature when excluding: weather


		f-string expression part cannot include a backslash (docments.py, line 359)
/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Test R2 score 0.5592
Test MAE score 0.0863
Test MSE score 0.0135
Test P95 score 0.2439
Test ECE score 0.0130
Evaluating feature when excluding: kinematic


...

Test R2 score 0.5634
Test MAE score 0.0856
Test MSE score 0.0134
Test P95 score 0.2382
Test ECE score 0.0126
Evaluating feature when excluding: temporal


		f-string expression part cannot include a backslash (docments.py, line 359)
/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Test R2 score 0.4898
Test MAE score 0.1063
Test MSE score 0.0195
Test P95 score 0.2827
Test ECE score 0.0113
Evaluating feature when excluding: temporal


		f-string expression part cannot include a backslash (docments.py, line 359)
/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Test R2 score 0.5597
Test MAE score 0.0860
Test MSE score 0.0135
Test P95 score 0.2411
Test ECE score 0.0111


# Delta Between Feature-groups


In [17]:

results_df = pd.DataFrame(delta_results)

results_df["target"] = results_df["target"].map({"iou": "IoU", "lrp": "LRP"})
results_df["delta_r2"] = results_df["delta_r2"].round(4)
results_df["delta_ece"] = results_df["delta_ece"].round(4)
results_df = results_df.sort_values(by="target")
display(results_df)


,target,feature_group,delta_r2,delta_ece
0,IoU,weather,-0.0009,-0.0012
2,IoU,kinematic,-0.0131,0.0005
4,IoU,calibration,-0.0004,0.0004
6,IoU,image_quality,0.0054,0.0009
8,IoU,model,-0.3760,0.0131
10,IoU,location,-0.0015,0.0015
12,IoU,temporal,-0.0059,-0.0008
1,LRP,weather,0.0635,0.0009
3,LRP,kinematic,0.0596,-0.0003
5,LRP,calibration,0.0713,0.0008


In [18]:
print(results_df.to_latex(index=False))

\begin{tabular}{llrr}
\toprule
target & feature_group & delta_r2 & delta_ece \\
\midrule
IoU & weather & -0.000900 & -0.001200 \\
IoU & kinematic & -0.013100 & 0.000500 \\
IoU & calibration & -0.000400 & 0.000400 \\
IoU & image_quality & 0.005400 & 0.000900 \\
IoU & model & -0.376000 & 0.013100 \\
IoU & location & -0.001500 & 0.001500 \\
IoU & temporal & -0.005900 & -0.000800 \\
LRP & weather & 0.063500 & 0.000900 \\
LRP & kinematic & 0.059600 & -0.000300 \\
LRP & calibration & 0.071300 & 0.000800 \\
LRP & image_quality & 0.073700 & -0.000700 \\
LRP & model & -0.394200 & 0.016400 \\
LRP & location & 0.067700 & 0.000500 \\
LRP & temporal & 0.064000 & -0.001000 \\
\bottomrule
\end{tabular}

